# 08 — Senescence across tissues

Senescent cells stop dividing for good and secrete an inflammatory mix (the SASP); they build up with age, and clearing them extends healthy lifespan in mice. No single gene marks them, so each cell is scored with **SenMayo** (Saul et al. 2022), a 117-gene mouse set curated for this (MSigDB `SAUL_SEN_MAYO`).

The question: in which tissue × cell type does the senescence score rise with age **per mouse**? Every row of the atlas is scored, so the robust rows (age signal, not technically confounded) can be compared with the others. Reads `figure/atlas/atlas.csv` from 07 and `data/senmayo_mouse.txt`.

### 1. Setup

In [ ]:
import sys
sys.path.append("..")          # so "src" can be imported from the notebook folder

import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import false_discovery_control, spearmanr
from src import aging as ag

A = ag.paths("atlas")
atlas = pd.read_csv(A.figure("atlas.csv"))
atlas["robust"] = atlas["verdict"].eq("age signal") & ~atlas["tech_confounded"].astype(bool)

### 2. Score every cell, one tissue at a time

`sc.tl.score_genes` gives each cell the mean expression of the SenMayo genes minus the mean of a random reference set **matched for expression level**, so a cell that simply has more RNA doesn't score higher. Per mouse and cell type it keeps: the mean score, the fraction of "SenMayo-high" cells (above the 95th percentile of that row's young cells), the mean of *Cdkn1a* (p21) and the fraction of cells with any *Cdkn2a* (p16; mostly dropout).

In [ ]:
rows = []
for tissue in sorted(atlas["tissue"].unique()):
    a = sc.read_h5ad(ag.paths(tissue).processed)
    genes, missing = ag.load_gene_set(ag.ROOT / "data" / "senmayo_mouse.txt", a)
    sc.tl.score_genes(a, genes, score_name="senmayo", random_state=0)
    obs = a.obs[["cell_ontology_class", "mouse.id", "age", "senmayo"]].copy()
    obs["cdkn1a"] = a[:, "Cdkn1a"].X.toarray().ravel()
    obs["cdkn2a"] = (a[:, "Cdkn2a"].X.toarray().ravel() > 0) if "Cdkn2a" in a.var_names else np.nan
    for ct in atlas.loc[atlas["tissue"] == tissue, "cell_type"]:
        drop = ag.main_kw(ct, tissue).get("drop_mice", [])
        o = obs[(obs["cell_ontology_class"] == ct) & ~obs["mouse.id"].isin(drop)
                & obs["age"].isin(ag.YOUNG + ag.OLD)]
        high = np.quantile(o.loc[o["age"].isin(ag.YOUNG), "senmayo"], 0.95)
        per = o.groupby("mouse.id", observed=True).agg(
            age=("age", "first"), n_cells=("senmayo", "size"), senmayo=("senmayo", "mean"),
            frac_high=("senmayo", lambda v: (v > high).mean()),
            cdkn1a=("cdkn1a", "mean"), cdkn2a_frac=("cdkn2a", "mean"))
        rows.append(per.reset_index().assign(tissue=tissue, cell_type=ct))
    print(tissue, f"{len(genes)} SenMayo genes used, missing: {missing}")
    del a
per_mouse = pd.concat(rows, ignore_index=True)
per_mouse["age"] = per_mouse["age"].astype(str)
per_mouse["months"] = per_mouse["age"].map(ag.AGE_MONTHS)
per_mouse = per_mouse[per_mouse["n_cells"] >= 10]          # a mouse mean from < 10 cells is noise
per_mouse.round(4).to_csv(A.figure("senescence_by_mouse.csv"), index=False)

### 3. Trend with age, per row

Spearman ρ between age in months and the per-mouse score; mice are the samples. Batch check for a single score: the trend is recomputed with **each age group left out in turn**, and `stable` says whether ρ keeps its sign every time, so one age batch can't make the trend on its own. `p_bh` corrects across all rows.

In [ ]:
def trend(d, col):
    rho, p = spearmanr(d["months"], d[col]) if d["months"].nunique() > 1 else (np.nan, np.nan)
    loo = [spearmanr(x["months"], x[col])[0] for a in d["age"].unique()
           if (x := d[d["age"] != a])["months"].nunique() > 1]
    young, old = d[d["age"].isin(ag.YOUNG)][col].mean(), d[d["age"].isin(ag.OLD)][col].mean()
    return pd.Series({"rho": rho, "p": p, "old_minus_young": old - young, "n_mice": len(d),
                      "stable": bool(len(loo)) and all(np.sign(r) == np.sign(rho) for r in loo)})

trends = []
for col in ["senmayo", "frac_high", "cdkn1a"]:
    t = per_mouse.groupby(["tissue", "cell_type"]).apply(trend, col=col, include_groups=False).reset_index()
    t["measure"] = col
    t["p_bh"] = false_discovery_control(t["p"].fillna(1))
    trends.append(t)
trends = pd.concat(trends).merge(atlas[["tissue", "cell_type", "compartment", "verdict", "robust",
                                        "cross_age_all_splits"]], on=["tissue", "cell_type"])
trends.round(4).to_csv(A.figure("senescence_trends.csv"), index=False)
sen = trends[trends["measure"] == "senmayo"].sort_values("rho", ascending=False)
sen[["tissue", "cell_type", "compartment", "robust", "rho", "p", "p_bh", "stable", "n_mice"]].round(3)

### 4. Robust rows vs. the rest

Does senescence rise more in cell types with a robust age signal? Counted as rows with ρ > 0, with ρ > 0 and stable, and with `p_bh` < 0.1.

In [ ]:
def summary(t):
    return pd.Series({"rows": len(t), "rho_up": (t["rho"] > 0).sum(),
                      "up_and_stable": ((t["rho"] > 0) & t["stable"]).sum(),
                      "up_bh_0.1": ((t["rho"] > 0) & (t["p_bh"] < 0.1)).sum(),
                      "median_rho": t["rho"].median()})

out = {}
for m in ["senmayo", "frac_high", "cdkn1a"]:
    t = trends[trends["measure"] == m]
    out[(m, "robust")] = summary(t[t["robust"]])
    out[(m, "other")] = summary(t[~t["robust"]])
    out[(m, "robust, structural")] = summary(t[t["robust"] & t["compartment"].isin(["stromal", "endothelial", "epithelial"])])
    out[(m, "immune")] = summary(t[t["compartment"] == "immune"])
pd.DataFrame(out).T.round(2)

### 5. Figure: senescence trend per row

One point per tissue × cell type: ρ of the SenMayo score with age, coloured by compartment, filled if the row is robust, ringed if the trend is stable.

In [ ]:
d = sen.assign(label=sen["tissue"] + ": " + sen["cell_type"].str.slice(0, 38)).sort_values("rho")
fig, ax = plt.subplots(figsize=(6, 0.22 * len(d) + 1))
pal = dict(zip(["stromal", "endothelial", "epithelial", "immune", "progenitor", "other"], sns.color_palette("tab10")))
for i, r in enumerate(d.itertuples()):
    ax.scatter(r.rho, i, s=45, color=pal[r.compartment] if r.robust else "white",
               edgecolor=pal[r.compartment], linewidth=2 if r.stable else 0.8, zorder=3)
ax.axvline(0, color="gray", lw=0.6)
ax.set_yticks(range(len(d))); ax.set_yticklabels(d["label"], fontsize=7)
ax.set_xlabel("Spearman rho, SenMayo score vs age (per mouse)")
for c, col in pal.items():
    ax.scatter([], [], color=col, label=c)
ax.legend(fontsize=7, loc="lower right", title="filled = robust age signal", title_fontsize=7)
fig.savefig(A.figure("senescence_trends.png"), dpi=200, bbox_inches="tight")

### 6. Per-mouse view for the stromal rows

In [ ]:
show = sen[sen["compartment"].eq("stromal")][["tissue", "cell_type"]]
d = per_mouse.merge(show, on=["tissue", "cell_type"])
d["row"] = d["tissue"] + "\n" + d["cell_type"].str.slice(0, 28)
g = sns.catplot(data=d, x="months", y="senmayo", col="row", col_wrap=4, kind="strip",
                height=2.6, sharey=False, size=6, jitter=0.1, native_scale=True)
g.set_titles("{col_name}", size=8)
g.savefig(A.figure("senescence_stromal_by_mouse.png"), dpi=200, bbox_inches="tight")

### 7. *Cdkn2a* (p16): how often is it detected at all?

p16 is the classic senescence marker but is barely captured by droplet sequencing. The fraction of cells with any count, per row, shows whether it can be used here.

In [ ]:
per_mouse.groupby(["tissue", "cell_type"])["cdkn2a_frac"].mean().describe().round(4)

### Findings

70 tissue × cell type rows, per mouse (≥ 10 cells), 9 tissues. 115 of the 117 SenMayo genes are in the atlas (*Bex3* under its old name *Ngfrap1*; *Ccl26* and *Spx* absent).

- **The SenMayo score does not track age.** In the 26 robust rows, 15 have ρ > 0 and none survives correction (median ρ 0.10); the 44 other rows look the same (median 0.09). The fraction of SenMayo-high cells is no better. Most SenMayo genes are secreted factors (chemokines, growth factors), weakly captured per cell, so a per-cell composite score dilutes the few that move.
- **p21 (*Cdkn1a*) on its own rises with age in structural cells across tissues.** In robust structural rows, 14 of 15 have ρ > 0 (median ρ 0.45; immune rows 0.16), and the rise appears in all 5 tissues that have such a row. Strongest, each stable with any one age group left out: bladder endothelium ρ 0.95 (BH 0.03), lung fibroblast 0.78 (0.04), lung adventitial 0.77 (0.09), kidney collecting duct 0.73 (0.05), tongue basal epithelium 0.72 (0.11), muscle endothelium 0.66 (0.06). Kidney contributes 7 rows from the same mice, so count it as one tissue: 5 of 5 tissues, not 14 independent results.
- **p16 (*Cdkn2a*) cannot be used:** detected in 0.8% of cells in the median row (max 11%), the dropout the guide expected.
- **Read as:** inside TMS, the cell-cycle-arrest arm of senescence (p21) rises in structural cells across organs, and the secretory arm (SASP) is not visible in droplet data at this depth.
- **But it does not replicate (10).** In Kimmel et al. 2019 (7–8 vs ~21 months, another lab), p21 is flat or lower in old lung fibroblast, adventitial and AT2 cells, and higher in old lung immune cells. The TMS rise survives removing dissociation stress, so the likeliest reading is timing: a rise between 3 and 7 months that neither dataset samples. p21 is reported as a TMS observation, not an aging marker.